# GPU Systems Deep Dive on Training
We will run a tiny transformer training loop and measure:

1. weights memory
2. gradients memory
3. optimizer state memory
4. activation memory

## Step 0 - load dataset
We will use **nmt dataset for english to spanish translation**

In [1]:
from datasets import load_dataset
import tokenizers

In [2]:
nmt_original_train_set, nmt_original_test_set = load_dataset(path="ageron/tatoeba_mt_train",
                                                             name="eng-spa",
                                                             split=["validation", "test"])
split = nmt_original_train_set.train_test_split(test_size=0.2, seed=42)
nmt_train_set = split["train"]
nmt_valid_set = split["test"]

README.md: 0.00B [00:00, ?B/s]

eng-spa/validation-00000-of-00001.parque(…):   0%|          | 0.00/7.85M [00:00<?, ?B/s]

eng-spa/test-00000-of-00001.parquet:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/197299 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24514 [00:00<?, ? examples/s]

This is the view of dataset, where we have source_text in english and target_text in spanish

In [3]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

## Step 0 - Tokenizers

Create BPE tokenizer, and train the tokenizer on both english and spanish text

In [4]:
def eng_to_spa():
  """Function to provide iterator for tokenizer"""
  for pair in nmt_train_set:
    yield pair["source_text"]
    yield pair["target_text"]

In [5]:
max_length=128
vocab_size=10_000

tokenizer_model = tokenizers.models.BPE(ink_token="<unk>")
tokenizer = tokenizers.Tokenizer(tokenizer_model)

tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
tokenizer.enable_truncation(max_length=max_length)
tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()

tokenizer_trainer = tokenizers.trainers.BpeTrainer(vocab_size=vocab_size,
                                                   special_tokens=["<unk>", "<pad>", "<s>", "</s>"])
tokenizer.train_from_iterator(eng_to_spa(), tokenizer_trainer)

In [6]:
tokenizer.encode(nmt_train_set[0]["source_text"]).ids

[217, 1826, 196, 1600, 444, 214, 3006, 17]

## Step 0 - DataLoader

Here we need collate_fn function.
- The collate_fn() function starts by extracting all the English and Spanish texts from the given batch.
- In the process, it also **adds an SoS token at the start of each Spanish text**, **as well as an EoS token at the end**.
- It then tokenizes both the English and Spanish texts will use BPE tokenizer, which we will also make and train on this same dataset in "Step 0 Tokenizers"
- Next, the input sequences and the attention masks are converted to tensors and wrapped in an NmtPair, which is named tuple
  - Importantly, **the function drops the EoS token from the decoder inputs**, and **drops the SoS token from the decoder targets**.
  - For example,
    - the inputs may contain the token IDs for “\<s> Me gusta el fútbol” but now \</s>,
    - while the targets may contain the token IDs for “Me gusta el fútbol \</s>” but not \<s>
- Lastly, the function returns the inputs (i.e., the NmtPair) along with the targets. Then we just create the data loaders as usual.

We will start with NmtPair Named Tuple, this is not necessary, but with the help of this errors can be avoided

In [7]:
import torch
import torch.nn as nn

In [8]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPair", fields)):
  def to(self, device):
    return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                   self.tgt_token_ids.to(device), self.tgt_mask.to(device))

In [9]:
def collate_fn(batch):
  """This is a collate function required by DataLoader"""
  src_texts = [pair["source_text"] for pair in batch]
  tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
  src_encodings = tokenizer.encode_batch(src_texts)
  tgt_encodings = tokenizer.encode_batch(tgt_texts)
  src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
  tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
  src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
  tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
  # remove </s> from tgt
  inputs = NmtPair(src_token_ids, src_mask,
                   tgt_token_ids[:, :-1], tgt_mask[:, :-1])
  labels = tgt_token_ids[:, 1:]
  return inputs, labels

In [10]:
from torch.utils.data import DataLoader

# change this for the experiment
batch_size=32

nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              shuffle=True, collate_fn=collate_fn)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=collate_fn)
nmt_test_loader = DataLoader(nmt_original_test_set, batch_size=batch_size,
                             collate_fn=collate_fn)

In [11]:
next(iter(nmt_train_loader))[0]

NmtPair(src_token_ids=tensor([[2181,  217,  196,  715,   64, 3249,   17,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0],
        [  43,  567,  315, 3442,  196,  529,  334,  224,   15,  512,   43,  557,
           10,   83,  315,  463,   17,    0,    0,    0,    0,    0,    0,    0,
            0],
        [  43,  393,  443,   43,  410,   17,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0],
        [3392,  453, 1300,  198,  365, 1067, 3026,   17,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0],
        [  43,  696, 5344,  245,   10,   82,  511,  559,  516,  293,   43,  315,
         1027,   17,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0],
        [ 489,  207,  235,  214, 5547,   17,    0,    0,    0,    0,    0,    0,
       

- We got first batch for training with 8 samples in it, which has src_token_ids, src_mask, tgt_token_ids, tgt_mask
- here pad_id=0 is applied,

## Step 1 — Create tiny transformer
Example:
- layers = 4
- hidden = 256
- heads = 4
- seq_len = 128
- Batch = 8

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
class PositionalEmbedding(nn.Module):
  def __init__(self, max_length=128, dim_size=256, dropout=0.1):
    super().__init__()
    self.pos_embed = nn.Parameter(torch.randn(max_length, dim_size) * 0.2)
    self.dropout = nn.Dropout(dropout)

  def forward(self, X):
    return self.dropout(X + self.pos_embed[:X.size(1)]) # till X lengths

In [14]:
# English to spanish translation transfomer
class NmtTransformer(nn.Module):
  def __init__(self, vocab_size, max_length=128, input_dim_size=512,
               n_heads=4, num_layers=4, pad_id=0, dropout=0.1):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, input_dim_size)
    self.pos_embed = PositionalEmbedding(max_length, input_dim_size, dropout)
    self.transformer = nn.Transformer(d_model=input_dim_size, nhead=n_heads,
                                      num_encoder_layers=num_layers,
                                      num_decoder_layers=num_layers,
                                      batch_first=True)
    self.output = nn.Linear(input_dim_size, vocab_size)

  def forward(self, pair):
    src_embed = self.pos_embed(self.embed(pair.src_token_ids))
    tgt_embed = self.pos_embed(self.embed(pair.tgt_token_ids))
    src_pad_mask = pair.src_mask.bool()
    tgt_pad_mask = pair.tgt_mask.bool()

    # create attnetion mask for decoder
    size = [pair.tgt_token_ids.size(1)] * 2 # Lk, Lk shape
    full_mask = torch.full(size, True, device=tgt_pad_mask.device)
    casual_mask = torch.triu(full_mask, diagonal=1)

    # transformer
    decoder_output = self.transformer(
        src_embed, tgt_embed,
        src_key_padding_mask=src_pad_mask,
        memory_key_padding_mask=src_pad_mask,
        tgt_mask=casual_mask,
        tgt_key_padding_mask=tgt_pad_mask
    )

    # Permute was performed for entropy loss and metric,
    # as they expect (batch_size, tgt_seq_len/class, hidden_size)
    return self.output(decoder_output).permute(0, 2, 1)

## Step 2 - Preapre evaluate and train functions

In [15]:
def print_gpu_memory(stage, epoch, batch_idx):
  """This will print GPU memory stats"""
  if epoch == 0 and batch_idx == 0:
    print(f"\n--- {stage} ---")
    print("allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
    print("reserved :", torch.cuda.memory_reserved() / 1024**2, "MB")
    print("max alloc:", torch.cuda.max_memory_allocated() / 1024**2, "MB")

In [16]:
def evaluate_tm(model, metric, data_loader):
  """Evaluation Function for valid dataset"""
  model.eval()
  metric.reset()
  for X_batch, y_batch in data_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    y_pred = model(X_batch)
    metric.update(y_pred, y_batch)
  return metric.compute().item()

In [17]:
def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs=10, patience=2, factor=0.5):
  """Funtion to train model"""
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
      optimizer, mode="max", patience=patience, factor=factor
  )
  history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
  for epoch in range(n_epochs):
    print(f"This is epoch: {epoch + 1}")
    model.train()
    metric.reset()
    total_loss = 0
    for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      print_gpu_memory("before Forward pass", epoch, batch_idx)
      y_pred = model(X_batch)
      print_gpu_memory("After forward pass", epoch, batch_idx)
      loss = loss_fn(y_pred, y_batch)
      total_loss += loss.item()
      print_gpu_memory("After computing loss", epoch, batch_idx)
      loss.backward()
      print_gpu_memory("After Backward pass", epoch, batch_idx)
      optimizer.step()
      print_gpu_memory("After Optimization", epoch, batch_idx)
      optimizer.zero_grad()
      metric.update(y_pred, y_batch)
    history["train_losses"].append(total_loss / len(train_loader))
    history["train_metrics"].append(metric.compute().item())
    valid_metric = evaluate_tm(model, metric, valid_loader)
    history["valid_metrics"].append(valid_metric)
    scheduler.step(valid_metric)
    print(f"\rEpoch {epoch + 1}/{n_epochs}, "
          f"train loss: {history['train_losses'][-1]:.4f}, "
          f"train metric: {history['train_metrics'][-1]:.2%}, "
          f"valid metric: {history['valid_metrics'][-1]:.2%}")
  return history

### Garbage collector

Lets also define garbage collector, this will free the GPU RAM regularly to avoid running out of space

For this, we will delete the models and tensors as we go, using the del_vars() function below. It deletes the given variables (if they exist) then calls Python's garbage collector, and it also calls torch.cuda.empty_cache() if we're using a CUDA GPU.

In [18]:
import gc
def del_vars(variable_names=[]):
  for var in variable_names:
    try:
      del globals()[var]
    except KeyError:
      pass
  # It forces Python to immediately free unused memory.
  gc.collect()

  # free gpu memory
  if device == "cuda":
    torch.cuda.empty_cache()

## Step 2 — Run training loop

- forward
- loss
- backward
- optimizer.step()

Step 3 — Measure GPU memory
- Track:
- torch.cuda.max_memory_allocated()
- torch.cuda.memory_allocated()
- torch.cuda.memory_reserved()

In [19]:
%pip install torchmetrics
import torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.3 MB/s eta 0:00:00


In [20]:
torch.manual_seed(42)
print_gpu_memory("Starting point", 0, 0)
nmt_model = NmtTransformer(vocab_size=vocab_size).to(device)
print_gpu_memory("After creating model", 0, 0)

optimizer = torch.optim.NAdam(nmt_model.parameters())
loss_fn = nn.CrossEntropyLoss(ignore_index=0) #gnore <pad> tokens
metric = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size).to(device)


--- Starting point ---
allocated: 0.0 MB
reserved : 0.0 MB
max alloc: 0.0 MB

--- After creating model ---
allocated: 152.54638671875 MB
reserved : 174.0 MB
max alloc: 152.54638671875 MB


In [21]:
nmt_model

NmtTransformer(
  (embed): Embedding(10000, 512)
  (pos_embed): PositionalEmbedding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-3): 4 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): TransformerDec

In [22]:
history = train(nmt_model, optimizer, loss_fn, metric,
                nmt_train_loader, nmt_valid_loader, n_epochs=1)

This is epoch: 1

--- before Forward pass ---
allocated: 152.580078125 MB
reserved : 174.0 MB
max alloc: 152.580078125 MB

--- After forward pass ---
allocated: 478.6748046875 MB
reserved : 498.0 MB
max alloc: 482.6787109375 MB

--- After computing loss ---
allocated: 510.67578125 MB
reserved : 562.0 MB
max alloc: 542.6748046875 MB

--- After Backward pass ---
allocated: 362.595703125 MB
reserved : 600.0 MB
max alloc: 574.67578125 MB

--- After Optimization ---
allocated: 672.5009765625 MB
reserved : 876.0 MB
max alloc: 828.04736328125 MB
Epoch 1/1, train loss: 6.3351, train metric: 3.72%, valid metric: 3.76%


In [23]:
from google.colab import drive
drive.mount('/content/drive')

torch.save(nmt_model.state_dict(), "/content/drive/MyDrive/models/nmt_trans_gpu_model.pt")

# nmt_model.load_state_dict(torch.load("/content/drive/MyDrive/models/nmt_trans_gpu_model.pt", map_location=torch.device('cpu')))

Mounted at /content/drive


## Step 3 - cleanup

In [24]:
del nmt_test_loader, nmt_train_loader, nmt_valid_loader, loss_fn
del metric, nmt_model, nmt_train_set, nmt_valid_set, optimizer
print_gpu_memory("Before Garbage Cleanup", 0, 0)
del_vars(device)
print_gpu_memory("After Garbage Cleanup", 0, 0)


--- Before Garbage Cleanup ---
allocated: 169.80078125 MB
reserved : 3266.0 MB
max alloc: 2611.3466796875 MB

--- After Garbage Cleanup ---
allocated: 169.79638671875 MB
reserved : 216.0 MB
max alloc: 2611.3466796875 MB


## Step 4 - Prediction

In [26]:
nmt_model = NmtTransformer(vocab_size=vocab_size).to(device)
nmt_model.load_state_dict(torch.load("/content/drive/MyDrive/models/nmt_trans_gpu_model.pt", map_location=torch.device('cpu')))

<All keys matched successfully>

In [46]:
def translate(src_txt, model, tokenizer, device, max_length=50):
    model.eval()

    src_enc  = tokenizer.encode(src_txt)
    src_ids  = torch.tensor([src_enc.ids]).to(device)
    src_mask = (~torch.tensor([src_enc.attention_mask], dtype=torch.bool)).to(device)

    bos_id = tokenizer.token_to_id("<s>")
    eos_id = tokenizer.token_to_id("</s>")
    pad_id = tokenizer.token_to_id("<pad>")

    tgt_ids = torch.tensor([[bos_id]]).to(device)

    with torch.no_grad():
        for step in range(max_length):
            tgt_mask = torch.zeros(tgt_ids.shape, dtype=torch.bool).to(device)

            pair = NmtPair(
                src_token_ids = src_ids,
                src_mask      = src_mask,
                tgt_token_ids = tgt_ids,
                tgt_mask      = tgt_mask,
            )

            logits      = model(pair)       # (1, vocab, tgt_len)
            last_logits = logits[0, :, -1]  # (vocab,)

            # Block <pad> and <eos> for all steps except last
            # This forces the model to generate real content
            last_logits[pad_id] = float('-inf')
            if step < 3:  # allow </s> only after at least 3 tokens
                last_logits[eos_id] = float('-inf')

            top5_tokens = [tokenizer.id_to_token(i.item()) for i in last_logits.topk(5).indices]
            print(f"Step {step+1}: {top5_tokens}")

            next_token = last_logits.argmax().unsqueeze(0)
            tgt_ids    = torch.cat([tgt_ids, next_token.unsqueeze(0)], dim=1)

            if next_token.item() == eos_id:
                break

    generated_ids = tgt_ids[0].tolist()
    clean_ids     = [t for t in generated_ids if t not in {bos_id, eos_id, pad_id}]
    return tokenizer.decode(clean_ids)


translation = translate("How was your day?", nmt_model, tokenizer, device)
print_gpu_memory("After prediction", 0, 0)
print("\nSource:     ", "How was your day?")
print("Translation:", translation)

Step 1: ['.', 'de', 'que', 'la', 'a']
Step 2: ['.', 'de', 'que', 'la', 'a']
Step 3: ['.', 'de', 'que', 'la', 'a']
Step 4: ['</s>', '.', 'de', 'que', 'la']

--- After prediction ---
allocated: 324.44189453125 MB
reserved : 500.0 MB
max alloc: 2611.3466796875 MB

Source:      How was your day?
Translation: . . .
